# 17 — Calibrating and Stress-Testing a World Model

## Learning outcome

Measure one-step prediction interval coverage under random actions and planner actions. Then compare a model-scored plan with the simulator result for the same action sequence. This lesson uses a state-based polynomial model in MountainCarContinuous. It is a small local analogue. It is not a JEPA model or a reproduction of a research paper.

## Prerequisites and output

You need Python 3.10–3.13 and the project dependencies. The experiment uses CPU only. It downloads no data or model weights. It writes four CSV evidence files under `docs/evidence/`.

## Fixed protocol

- Fit five independently seeded ensembles. Each ensemble has five bootstrap members.
- Use ten 200-step random-action episodes for each model run. Keep calibration, random-action test, planner-action test, and final evaluation seeds disjoint.
- Calibrate separate 90% position and velocity intervals on the calibration episodes only.
- Use the cross-entropy method (CEM), a sampling-based planner, with horizon 8, 12 candidates, 3 elites, and 2 updates. Compare mean-return CEM with a fixed `mean - 1 × return standard deviation` score.
- Evaluate both learned planners on the same 20 starts for every model run. Evaluate the direct momentum rule once on the same starts.
- Report episode and model-run records. Any missing or failed item remains visible. Intervals across five model runs are uncertain.

The environment uses a 200-step cap to keep the lesson local and bounded. The official Gymnasium environment defaults to a 999-step cap.

In [1]:
import csv
import json
import platform
import sys
import time
from datetime import datetime, timezone
from pathlib import Path
from time import perf_counter

import gymnasium as gym
import numpy as np
import scipy
from scipy.stats import t as student_t
import sklearn
from sklearn.linear_model import Ridge
from sklearn.preprocessing import PolynomialFeatures

ENV_ID = "MountainCarContinuous-v0"
MAX_STEPS = 200
N_MODEL_RUNS = 5
N_BOOTSTRAP_MEMBERS = 5
N_TRAIN_EPISODES = 10
N_EVAL_STARTS = 20
INTERVAL_LEVEL = 0.90
POLY_DEGREE = 3
RIDGE_ALPHA = 1e-6
CEM_HORIZON = 8
CEM_CANDIDATES = 12
CEM_ELITES = 3
CEM_UPDATES = 2
RISK_COEFFICIENT = 1.0
EPSILON = 1e-8
def locate_repo_root():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / ".specify" / "feature.json").is_file() and (candidate / "pyproject.toml").is_file():
            return candidate
    raise RuntimeError("Run this notebook from inside the project repository")


REPO_ROOT = locate_repo_root()
OUTPUT_DIR = REPO_ROOT / "docs" / "evidence"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
NOTEBOOK_STARTED = perf_counter()

SEED_GROUPS = {
    "training": [list(range(1000 + 100 * run_index, 1010 + 100 * run_index)) for run_index in range(N_MODEL_RUNS)],
    "calibration": list(range(2000, 2010)),
    "in_distribution_test": list(range(3000, 3010)),
    "policy_shift_test": list(range(3500, 3510)),
    "final_evaluation": list(range(4000, 4020)),
}
all_episode_seeds = [seed for group in SEED_GROUPS.values() for item in group for seed in (item if isinstance(item, list) else [item])]
assert len(all_episode_seeds) == len(set(all_episode_seeds)), "Episode seed groups overlap"
assert all(len(group) == N_TRAIN_EPISODES for group in SEED_GROUPS["training"])
assert len(SEED_GROUPS["final_evaluation"]) == N_EVAL_STARTS

SOFTWARE_VERSIONS = {
    "python": sys.version.split()[0],
    "numpy": np.__version__,
    "scipy": scipy.__version__,
    "scikit_learn": sklearn.__version__,
    "gymnasium": gym.__version__,
}
print("Software versions:", SOFTWARE_VERSIONS)
print("Evidence directory:", OUTPUT_DIR)
print("Episode seeds are disjoint:", True)
print("Run count:", N_MODEL_RUNS, "; evaluation starts per run:", N_EVAL_STARTS)

Software versions: {'python': '3.13.14', 'numpy': '2.2.6', 'scipy': '1.15.3', 'scikit_learn': '1.7.2', 'gymnasium': '0.29.1'}
Evidence directory: /home/oscar/projects/education/jepa-world-models-tutorial/docs/evidence
Episode seeds are disjoint: True
Run count: 5 ; evaluation starts per run: 20


In [2]:
def make_env():
    return gym.make(ENV_ID, max_episode_steps=MAX_STEPS)


def direct_momentum_action(state):
    """Push with current momentum; near rest, push away from the goal side."""
    position, velocity = map(float, state)
    if velocity > 0.001:
        return 1.0
    if velocity < -0.001:
        return -1.0
    return -1.0 if position < 0.0 else 1.0


def collect_episode(seed, policy="random", ensemble=None, run_index=0):
    env = make_env()
    episode_started = perf_counter()
    try:
        state, _ = env.reset(seed=int(seed))
        env.action_space.seed(int(seed))
        states, actions, next_states = [], [], []
        episode_return = 0.0
        model_calls = 0
        planner_seconds = 0.0
        terminated = False
        truncated = False
        for step_index in range(MAX_STEPS):
            if policy == "random":
                action = float(env.action_space.sample()[0])
            elif policy == "direct_momentum":
                action = direct_momentum_action(state)
            elif policy in ("cem_mean", "cem_risk"):
                plan_seed = 1_000_000 + run_index * 100_000 + int(seed) * 10 + step_index + (29 if policy == "cem_risk" else 17)
                plan = cem_select(ensemble, state, policy, plan_seed)
                action = float(plan["sequence"][0])
                model_calls += plan["model_calls"]
                planner_seconds += plan["planner_seconds"]
            else:
                raise ValueError(f"Unknown policy: {policy}")

            next_state, reward, terminated, truncated, _ = env.step(np.asarray([action], dtype=np.float32))
            states.append(np.asarray(state, dtype=float))
            actions.append(action)
            next_states.append(np.asarray(next_state, dtype=float))
            episode_return += float(reward)
            state = next_state
            if terminated or truncated:
                break
        return {
            "seed": int(seed),
            "states": np.asarray(states, dtype=float).reshape(-1, 2),
            "actions": np.asarray(actions, dtype=float),
            "next_states": np.asarray(next_states, dtype=float).reshape(-1, 2),
            "episode_return": float(episode_return),
            "success": bool(terminated),
            "steps": len(actions),
            "model_calls": int(model_calls),
            "planner_seconds": float(planner_seconds),
            "elapsed_seconds": perf_counter() - episode_started,
            "status": "complete",
            "failure_detail": "",
        }
    finally:
        env.close()


def concatenate_transitions(episodes):
    return (
        np.concatenate([episode["states"] for episode in episodes], axis=0),
        np.concatenate([episode["actions"] for episode in episodes], axis=0),
        np.concatenate([episode["next_states"] for episode in episodes], axis=0),
    )


def fit_ensemble(states, actions, next_states, bootstrap_seeds):
    started = perf_counter()
    inputs = np.column_stack((states, actions))
    polynomial = PolynomialFeatures(degree=POLY_DEGREE, include_bias=True)
    features = polynomial.fit_transform(inputs)
    weights, biases = [], []
    for bootstrap_seed in bootstrap_seeds:
        rng = np.random.default_rng(int(bootstrap_seed))
        sample_indices = rng.integers(0, len(inputs), size=len(inputs))
        model = Ridge(alpha=RIDGE_ALPHA, fit_intercept=False)
        model.fit(features[sample_indices], next_states[sample_indices])
        weights.append(np.asarray(model.coef_, dtype=float))
        biases.append(np.zeros(2, dtype=float))  # fit_intercept=False defines a zero vector offset.
    bundle = {
        "polynomial": polynomial,
        "weights": np.stack(weights),
        "biases": np.stack(biases),
        "n_members": len(weights),
    }
    return bundle, perf_counter() - started


def predict_ensemble(bundle, states, actions):
    inputs = np.column_stack((states, actions))
    features = bundle["polynomial"].transform(inputs)
    return np.einsum("nf,mof->mno", features, bundle["weights"]) + bundle["biases"][:, None, :]


def rollout_candidate_returns(bundle, start_state, action_sequences):
    member_count = bundle["n_members"]
    candidate_count, horizon = action_sequences.shape
    states = np.broadcast_to(np.asarray(start_state, dtype=float), (member_count, candidate_count, 2)).copy()
    alive = np.ones((member_count, candidate_count), dtype=bool)
    returns = np.zeros((member_count, candidate_count), dtype=float)
    for step_index in range(horizon):
        actions = np.broadcast_to(action_sequences[:, step_index], (member_count, candidate_count))
        inputs = np.column_stack((states.reshape(-1, 2), actions.reshape(-1)))
        features = bundle["polynomial"].transform(inputs).reshape(member_count, candidate_count, -1)
        predicted = np.einsum("mkf,mof->mko", features, bundle["weights"]) + bundle["biases"][:, None, :]
        predicted[:, :, 0] = np.clip(predicted[:, :, 0], -1.2, 0.6)
        predicted[:, :, 1] = np.clip(predicted[:, :, 1], -0.07, 0.07)
        reached_goal = predicted[:, :, 0] >= 0.45
        step_reward = -0.1 * np.square(actions) + 100.0 * reached_goal
        returns += alive * step_reward
        alive &= ~reached_goal
        states = predicted
    return returns


def cem_select(bundle, start_state, method, seed):
    started = perf_counter()
    rng = np.random.default_rng(int(seed))
    mean = np.zeros(CEM_HORIZON, dtype=float)
    standard_deviation = np.full(CEM_HORIZON, 0.7, dtype=float)
    best_sequence = None
    best_mean_return = None
    best_return_std = None
    best_score = None
    for _ in range(CEM_UPDATES):
        candidates = np.clip(
            rng.normal(mean, standard_deviation, size=(CEM_CANDIDATES, CEM_HORIZON)),
            -1.0,
            1.0,
        )
        member_returns = rollout_candidate_returns(bundle, start_state, candidates)
        mean_returns = member_returns.mean(axis=0)
        return_stds = member_returns.std(axis=0)
        scores = mean_returns if method == "cem_mean" else mean_returns - RISK_COEFFICIENT * return_stds
        elite_indices = np.argsort(scores)[-CEM_ELITES:]
        best_index = int(np.argmax(scores))
        best_sequence = candidates[best_index].copy()
        best_mean_return = float(mean_returns[best_index])
        best_return_std = float(return_stds[best_index])
        best_score = float(scores[best_index])
        mean = candidates[elite_indices].mean(axis=0)
        standard_deviation = np.maximum(candidates[elite_indices].std(axis=0), 0.05)
    return {
        "sequence": best_sequence,
        "predicted_mean_return": best_mean_return,
        "ensemble_return_std": best_return_std,
        "planner_score": best_score,
        "model_calls": CEM_UPDATES * CEM_CANDIDATES * CEM_HORIZON * bundle["n_members"],
        "planner_seconds": perf_counter() - started,
    }


def simulate_open_loop(start_state, seed, sequence):
    env = make_env()
    try:
        env.reset(seed=int(seed))
        env.unwrapped.state = np.asarray(start_state, dtype=np.float64).copy()
        episode_return = 0.0
        steps = 0
        for action in sequence:
            _, reward, terminated, truncated, _ = env.step(np.asarray([float(action)], dtype=np.float32))
            episode_return += float(reward)
            steps += 1
            if terminated or truncated:
                break
        return float(episode_return), steps
    finally:
        env.close()


def student_t_summary(values):
    values = np.asarray(list(values), dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return {"n": 0, "mean": np.nan, "median": np.nan, "lower_95": np.nan, "upper_95": np.nan}
    mean_value = float(values.mean())
    if len(values) == 1:
        lower, upper = np.nan, np.nan
    else:
        half_width = float(student_t.ppf(0.975, df=len(values) - 1) * values.std(ddof=1) / np.sqrt(len(values)))
        lower, upper = mean_value - half_width, mean_value + half_width
    return {
        "n": int(len(values)),
        "mean": mean_value,
        "median": float(np.median(values)),
        "lower_95": lower,
        "upper_95": upper,
    }


def write_csv(path, rows, fieldnames):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)

## Fit independent models

Each run fits five degree-three polynomial Ridge models. A bootstrap sample draws rows with replacement from that run's ten training episodes. The ensemble spread is a disagreement score. It is not a probability until the interval scale is fit on the separate calibration split.

In [3]:
model_runs = []
for run_index in range(N_MODEL_RUNS):
    model_seed = 50_000 + run_index
    training_episode_seeds = SEED_GROUPS["training"][run_index]
    bootstrap_seeds = [60_000 + 100 * run_index + member_index for member_index in range(N_BOOTSTRAP_MEMBERS)]
    run_started = perf_counter()
    run_record = {
        "run_index": run_index,
        "model_seed": model_seed,
        "training_episode_seeds": training_episode_seeds,
        "bootstrap_seeds": bootstrap_seeds,
        "training_episode_count": len(training_episode_seeds),
        "training_sample_count": 0,
        "training_collection_seconds": 0.0,
        "model_fit_seconds": 0.0,
        "elapsed_seconds": 0.0,
        "training_elapsed_seconds": 0.0,
        "calibration_seconds": 0.0,
        "final_evaluation_seconds": 0.0,
        "status": "training_failed",
        "error_message": "",
        "bundle": None,
        "calibration_status": "not_run",
    }
    try:
        collection_started = perf_counter()
        training_episodes = [collect_episode(seed, policy="random") for seed in training_episode_seeds]
        states, actions, next_states = concatenate_transitions(training_episodes)
        run_record["training_collection_seconds"] = perf_counter() - collection_started
        run_record["training_sample_count"] = len(actions)
        bundle, fit_seconds = fit_ensemble(states, actions, next_states, bootstrap_seeds)
        run_record["model_fit_seconds"] = fit_seconds
        run_record["status"] = "training_complete"
        run_record["bundle"] = bundle
    except Exception as exc:
        run_record["error_message"] = f"{type(exc).__name__}: {exc}"
    run_record["elapsed_seconds"] = perf_counter() - run_started
    run_record["training_elapsed_seconds"] = run_record["elapsed_seconds"]
    model_runs.append(run_record)

print("Training outcomes:")
for record in model_runs:
    print({key: record[key] for key in ("model_seed", "training_sample_count", "training_collection_seconds", "model_fit_seconds", "status", "error_message")})

Training outcomes:
{'model_seed': 50000, 'training_sample_count': 2000, 'training_collection_seconds': 0.054881707997992635, 'model_fit_seconds': 0.004072322997672018, 'status': 'training_complete', 'error_message': ''}
{'model_seed': 50001, 'training_sample_count': 2000, 'training_collection_seconds': 0.05214777400396997, 'model_fit_seconds': 0.0032249560026684776, 'status': 'training_complete', 'error_message': ''}
{'model_seed': 50002, 'training_sample_count': 2000, 'training_collection_seconds': 0.05277294100233121, 'model_fit_seconds': 0.003706168005010113, 'status': 'training_complete', 'error_message': ''}
{'model_seed': 50003, 'training_sample_count': 2000, 'training_collection_seconds': 0.056329644001380075, 'model_fit_seconds': 0.003270140994573012, 'status': 'training_complete', 'error_message': ''}
{'model_seed': 50004, 'training_sample_count': 2000, 'training_collection_seconds': 0.06419717500102706, 'model_fit_seconds': 0.003311159001896158, 'status': 'training_complete',

## Calibrate and check one-step intervals

For each component, the scale is the 90th percentile of the absolute calibration residual divided by ensemble disagreement. The scale is then fixed. We measure it on the calibration split, new random-action episodes, and planner-action episodes. Only the calibration split sets the scale.

In [4]:
def transition_rows(episodes):
    rows = []
    for episode in episodes:
        for step_index, (state, action, next_state) in enumerate(zip(episode["states"], episode["actions"], episode["next_states"])):
            rows.append((episode["seed"], step_index, state, action, next_state))
    return rows


def make_calibration_records(model_seed, split, episodes, bundle, scales):
    raw = transition_rows(episodes)
    states = np.asarray([row[2] for row in raw], dtype=float)
    actions = np.asarray([row[3] for row in raw], dtype=float)
    actual = np.asarray([row[4] for row in raw], dtype=float)
    predictions = predict_ensemble(bundle, states, actions)
    means = predictions.mean(axis=0)
    deviations = predictions.std(axis=0)
    records = []
    for row_index, (episode_seed, step_index, state, action, next_state) in enumerate(raw):
        for component_index, component in enumerate(("position", "velocity")):
            deviation = float(deviations[row_index, component_index])
            scale = float(scales[component_index])
            mean_value = float(means[row_index, component_index])
            actual_value = float(actual[row_index, component_index])
            half_width = scale * deviation
            low = mean_value - half_width
            high = mean_value + half_width
            records.append({
                "model_seed": model_seed,
                "split": split,
                "episode_seed": episode_seed,
                "step": step_index,
                "component": component,
                "action": float(action),
                "previous_position": float(state[0]),
                "previous_velocity": float(state[1]),
                "observed_next_state": actual_value,
                "ensemble_mean": mean_value,
                "ensemble_std": deviation,
                "interval_scale": scale,
                "interval_low": low,
                "interval_high": high,
                "covered": bool(low <= actual_value <= high),
                "interval_width": float(2.0 * half_width),
            })
    return records


shared_calibration_episodes = [collect_episode(seed) for seed in SEED_GROUPS["calibration"]]
shared_random_test_episodes = [collect_episode(seed) for seed in SEED_GROUPS["in_distribution_test"]]
calibration_records = []
for run_record in model_runs:
    calibration_started = perf_counter()
    bundle = run_record["bundle"]
    if bundle is None:
        run_record["calibration_status"] = "missing_model"
        run_record["calibration_seconds"] = 0.0
        continue
    try:
        calibration_data = transition_rows(shared_calibration_episodes)
        calibration_states = np.asarray([row[2] for row in calibration_data], dtype=float)
        calibration_actions = np.asarray([row[3] for row in calibration_data], dtype=float)
        calibration_next = np.asarray([row[4] for row in calibration_data], dtype=float)
        calibration_predictions = predict_ensemble(bundle, calibration_states, calibration_actions)
        calibration_mean = calibration_predictions.mean(axis=0)
        calibration_std = calibration_predictions.std(axis=0)
        normalized_residuals = np.abs(calibration_next - calibration_mean) / np.maximum(calibration_std, EPSILON)
        scales = np.quantile(normalized_residuals, INTERVAL_LEVEL, axis=0)
        calibration_records.extend(make_calibration_records(run_record["model_seed"], "calibration", shared_calibration_episodes, bundle, scales))
        calibration_records.extend(make_calibration_records(run_record["model_seed"], "in_distribution_test", shared_random_test_episodes, bundle, scales))
        shift_episodes = [collect_episode(seed, policy="cem_mean", ensemble=bundle, run_index=run_record["run_index"]) for seed in SEED_GROUPS["policy_shift_test"]]
        calibration_records.extend(make_calibration_records(run_record["model_seed"], "policy_shift_test", shift_episodes, bundle, scales))
        run_record["calibration_scales"] = scales.tolist()
        run_record["calibration_status"] = "complete"
    except Exception as exc:
        run_record["calibration_status"] = "failed"
        run_record["error_message"] = (run_record["error_message"] + " | " if run_record["error_message"] else "") + f"calibration {type(exc).__name__}: {exc}"
    run_record["calibration_seconds"] = perf_counter() - calibration_started

calibration_summary_rows = []
for split in ("calibration", "in_distribution_test", "policy_shift_test"):
    for component in ("position", "velocity"):
        run_coverages, run_widths = [], []
        for run_record in model_runs:
            subset = [row for row in calibration_records if row["model_seed"] == run_record["model_seed"] and row["split"] == split and row["component"] == component]
            if subset:
                run_coverages.append(float(np.mean([row["covered"] for row in subset])))
                run_widths.append(float(np.mean([row["interval_width"] for row in subset])))
        coverage_stats = student_t_summary(run_coverages)
        width_stats = student_t_summary(run_widths)
        calibration_summary_rows.append({
            "split": split,
            "component": component,
            "runs": coverage_stats["n"],
            "coverage_mean": coverage_stats["mean"],
            "coverage_median": coverage_stats["median"],
            "coverage_lower_95": coverage_stats["lower_95"],
            "coverage_upper_95": coverage_stats["upper_95"],
            "mean_interval_width": width_stats["mean"],
        })
print("Calibration summary across independent model runs (95% Student-t intervals):")
for row in calibration_summary_rows:
    print(row)

Calibration summary across independent model runs (95% Student-t intervals):
{'split': 'calibration', 'component': 'position', 'runs': 5, 'coverage_mean': 0.9, 'coverage_median': 0.9, 'coverage_lower_95': 0.9, 'coverage_upper_95': 0.9, 'mean_interval_width': 4.5856222624984275e-06}
{'split': 'calibration', 'component': 'velocity', 'runs': 5, 'coverage_mean': 0.9, 'coverage_median': 0.9, 'coverage_lower_95': 0.9, 'coverage_upper_95': 0.9, 'mean_interval_width': 4.409231217725155e-06}
{'split': 'in_distribution_test', 'component': 'position', 'runs': 5, 'coverage_mean': 0.9071999999999999, 'coverage_median': 0.9085, 'coverage_lower_95': 0.8970308552390821, 'coverage_upper_95': 0.9173691447609177, 'mean_interval_width': 3.7647495789873696e-06}
{'split': 'in_distribution_test', 'component': 'velocity', 'runs': 5, 'coverage_mean': 0.9034000000000001, 'coverage_median': 0.9, 'coverage_lower_95': 0.8944742579519573, 'coverage_upper_95': 0.9123257420480428, 'mean_interval_width': 3.61805498420

## Compare plans and closed-loop policies

At each simulator step, each learned controller replans and executes the first action. For the initial selected plan at each start, a fresh simulator receives the same starting state and the exact selected action sequence. The signed gap is `model-predicted return − simulator return`. A positive gap means the model overpredicts that plan.

The direct momentum rule uses only the simulator state. It runs once per evaluation seed. It does not depend on a learned model.

In [5]:
def evaluate_learned_episode(bundle, model_seed, run_index, evaluation_seed, method):
    env = make_env()
    episode_started = perf_counter()
    plan_record = None
    planner_seconds = 0.0
    model_calls = 0
    episode_return = 0.0
    terminated = False
    truncated = False
    steps = 0
    status = "complete"
    failure_detail = ""
    try:
        state, _ = env.reset(seed=int(evaluation_seed))
        initial_state = np.asarray(state, dtype=float).copy()
        first_plan = cem_select(bundle, state, method, 2_000_000 + run_index * 100_000 + int(evaluation_seed) * 10 + (29 if method == "cem_risk" else 17))
        first_plan_return, first_plan_steps = simulate_open_loop(initial_state, evaluation_seed, first_plan["sequence"])
        planner_seconds += first_plan["planner_seconds"]
        model_calls += first_plan["model_calls"]
        plan_record = {
            "model_seed": model_seed,
            "evaluation_seed": int(evaluation_seed),
            "method": method,
            "start_position": float(initial_state[0]),
            "start_velocity": float(initial_state[1]),
            "action_sequence": json.dumps([float(value) for value in first_plan["sequence"]]),
            "predicted_mean_return": first_plan["predicted_mean_return"],
            "ensemble_return_std": first_plan["ensemble_return_std"],
            "planner_score": first_plan["planner_score"],
            "simulator_open_loop_return": first_plan_return,
            "simulator_open_loop_steps": first_plan_steps,
            "exploitability_gap": first_plan["predicted_mean_return"] - first_plan_return,
            "planner_model_calls": first_plan["model_calls"],
            "planner_seconds": first_plan["planner_seconds"],
            "status": "complete",
            "failure_detail": "",
        }
        state = initial_state
        for step_index in range(MAX_STEPS):
            if step_index == 0:
                plan = first_plan
            else:
                plan_seed = 2_000_000 + run_index * 100_000 + int(evaluation_seed) * 10 + step_index + (29 if method == "cem_risk" else 17)
                plan = cem_select(bundle, state, method, plan_seed)
                planner_seconds += plan["planner_seconds"]
                model_calls += plan["model_calls"]
            action = float(plan["sequence"][0])
            next_state, reward, terminated, truncated, _ = env.step(np.asarray([action], dtype=np.float32))
            episode_return += float(reward)
            steps += 1
            state = next_state
            if terminated or truncated:
                break
    except Exception as exc:
        status = "failed"
        failure_detail = f"{type(exc).__name__}: {exc}"
    finally:
        env.close()
    episode_record = {
        "method": method,
        "model_seed": model_seed,
        "evaluation_seed": int(evaluation_seed),
        "status": status,
        "success": bool(terminated) if status == "complete" else "",
        "episode_return": float(episode_return) if status == "complete" else "",
        "steps": steps,
        "model_calls": model_calls,
        "planner_seconds": planner_seconds,
        "elapsed_seconds": perf_counter() - episode_started,
        "failure_detail": failure_detail,
    }
    if plan_record is None:
        plan_record = {
            "model_seed": model_seed,
            "evaluation_seed": int(evaluation_seed),
            "method": method,
            "start_position": "",
            "start_velocity": "",
            "action_sequence": "",
            "predicted_mean_return": "",
            "ensemble_return_std": "",
            "planner_score": "",
            "simulator_open_loop_return": "",
            "simulator_open_loop_steps": "",
            "exploitability_gap": "",
            "planner_model_calls": "",
            "planner_seconds": "",
            "status": "failed",
            "failure_detail": failure_detail,
        }
    return episode_record, plan_record


def failure_episode_row(method, model_seed, evaluation_seed, detail):
    return {
        "method": method, "model_seed": model_seed, "evaluation_seed": int(evaluation_seed),
        "status": "not_run_model_failure", "success": "", "episode_return": "", "steps": 0,
        "model_calls": 0, "planner_seconds": 0.0, "elapsed_seconds": 0.0, "failure_detail": detail,
    }


def failure_plan_row(method, model_seed, evaluation_seed, detail):
    return {
        "model_seed": model_seed, "evaluation_seed": int(evaluation_seed), "method": method,
        "start_position": "", "start_velocity": "", "action_sequence": "",
        "predicted_mean_return": "", "ensemble_return_std": "", "planner_score": "",
        "simulator_open_loop_return": "", "simulator_open_loop_steps": "", "exploitability_gap": "",
        "planner_model_calls": "", "planner_seconds": "", "status": "not_run_model_failure", "failure_detail": detail,
    }


episode_records = []
plan_records = []
# The direct state rule has one result for each shared evaluation start.
for evaluation_seed in SEED_GROUPS["final_evaluation"]:
    try:
        direct = collect_episode(evaluation_seed, policy="direct_momentum")
        episode_records.append({
            "method": "direct_momentum", "model_seed": "", "evaluation_seed": int(evaluation_seed),
            "status": direct["status"], "success": direct["success"], "episode_return": direct["episode_return"],
            "steps": direct["steps"], "model_calls": 0, "planner_seconds": 0.0,
            "elapsed_seconds": direct.get("elapsed_seconds", ""), "failure_detail": direct["failure_detail"],
        })
    except Exception as exc:
        episode_records.append(failure_episode_row("direct_momentum", "", evaluation_seed, f"{type(exc).__name__}: {exc}"))

for run_record in model_runs:
    evaluation_started = perf_counter()
    bundle = run_record["bundle"]
    model_seed = run_record["model_seed"]
    for method in ("cem_mean", "cem_risk"):
        for evaluation_seed in SEED_GROUPS["final_evaluation"]:
            if bundle is None:
                episode_records.append(failure_episode_row(method, model_seed, evaluation_seed, run_record["error_message"] or "Model run unavailable"))
                plan_records.append(failure_plan_row(method, model_seed, evaluation_seed, run_record["error_message"] or "Model run unavailable"))
                continue
            try:
                episode_row, plan_row = evaluate_learned_episode(bundle, model_seed, run_record["run_index"], evaluation_seed, method)
            except Exception as exc:
                detail = f"{type(exc).__name__}: {exc}"
                episode_row = failure_episode_row(method, model_seed, evaluation_seed, detail)
                plan_row = failure_plan_row(method, model_seed, evaluation_seed, detail)
            episode_records.append(episode_row)
            plan_records.append(plan_row)
    run_record["final_evaluation_seconds"] = perf_counter() - evaluation_started

for run_record in model_runs:
    run_seed = run_record["model_seed"]
    expected = [row for row in episode_records if row["model_seed"] == run_seed]
    calibration_ok = run_record["calibration_status"] == "complete"
    eval_ok = len(expected) == 2 * N_EVAL_STARTS and all(row["status"] == "complete" for row in expected)
    plans_ok = len([row for row in plan_records if row["model_seed"] == run_seed]) == 2 * N_EVAL_STARTS and all(row["status"] == "complete" for row in plan_records if row["model_seed"] == run_seed)
    run_record["status"] = "complete" if run_record["bundle"] is not None and calibration_ok and eval_ok and plans_ok else "partial"
    run_record["elapsed_seconds"] = run_record["training_elapsed_seconds"] + run_record["calibration_seconds"] + run_record["final_evaluation_seconds"]

print("Evaluation row counts:", {
    "episodes": len(episode_records),
    "plans": len(plan_records),
    "calibration_component_rows": len(calibration_records),
})
print("Episode statuses:", {status: sum(row["status"] == status for row in episode_records) for status in sorted({row["status"] for row in episode_records})})
print("Model-run statuses:", [(row["model_seed"], row["status"], row["calibration_status"]) for row in model_runs])

Evaluation row counts: {'episodes': 220, 'plans': 200, 'calibration_component_rows': 60000}
Episode statuses: {'complete': 220}
Model-run statuses: [(50000, 'complete', 'complete'), (50001, 'complete', 'complete'), (50002, 'complete', 'complete'), (50003, 'complete', 'complete'), (50004, 'complete', 'complete')]


## Summarize and save evidence

The notebook reports a mean, median, and 95% Student-t interval across independent model runs. For five runs, the interval has four degrees of freedom and is uncertain. The direct policy has one set of 20 starts, so it has no interval across model runs. Paired differences compare each learned policy with the direct outcome on the same seed.

In [6]:
def metric_rows_for_method(method, field):
    if method == "direct_momentum":
        rows = [row for row in episode_records if row["method"] == method and row["status"] == "complete"]
        values = [float(row[field]) for row in rows if row[field] != ""]
        return {
            "observed_episodes": len(values),
            "expected_episodes": N_EVAL_STARTS,
            "mean_return": float(np.mean(values)) if values else np.nan,
            "median_return": float(np.median(values)) if values else np.nan,
        }
    run_values = []
    missing_rows = 0
    for run_record in model_runs:
        rows = [row for row in episode_records if row["method"] == method and row["model_seed"] == run_record["model_seed"]]
        valid = [row for row in rows if row["status"] == "complete" and row[field] != ""]
        missing_rows += N_EVAL_STARTS - len(valid)
        if valid:
            run_values.append(float(np.mean([float(row[field]) for row in valid])))
    summary = student_t_summary(run_values)
    return {"observed_runs": summary["n"], "missing_episode_rows": missing_rows, **summary}


print("Closed-loop return: per-run means, then summary across model runs")
for method in ("cem_mean", "cem_risk", "direct_momentum"):
    print(method, metric_rows_for_method(method, "episode_return"))

print("Observed success rate")
for method in ("cem_mean", "cem_risk", "direct_momentum"):
    if method == "direct_momentum":
        rows = [row for row in episode_records if row["method"] == method and row["status"] == "complete"]
        observed_rate = float(np.mean([bool(row["success"]) for row in rows])) if rows else np.nan
        print(method, {"observed_episodes": len(rows), "expected_episodes": N_EVAL_STARTS, "success_rate": observed_rate})
    else:
        run_rates = []
        missing = 0
        for run_record in model_runs:
            rows = [row for row in episode_records if row["method"] == method and row["model_seed"] == run_record["model_seed"]]
            valid = [row for row in rows if row["status"] == "complete"]
            missing += N_EVAL_STARTS - len(valid)
            if valid:
                run_rates.append(float(np.mean([bool(row["success"]) for row in valid])))
        print(method, {"missing_episode_rows": missing, **student_t_summary(run_rates)})

print("Paired return differences (learned method minus direct momentum, same evaluation seed)")
direct_by_seed = {row["evaluation_seed"]: row for row in episode_records if row["method"] == "direct_momentum" and row["status"] == "complete"}
for method in ("cem_mean", "cem_risk"):
    run_differences = []
    paired_episode_count = 0
    for run_record in model_runs:
        by_seed = {row["evaluation_seed"]: row for row in episode_records if row["method"] == method and row["model_seed"] == run_record["model_seed"] and row["status"] == "complete"}
        paired = [float(by_seed[seed]["episode_return"]) - float(direct_by_seed[seed]["episode_return"]) for seed in SEED_GROUPS["final_evaluation"] if seed in by_seed and seed in direct_by_seed]
        paired_episode_count += len(paired)
        if paired:
            run_differences.append(float(np.mean(paired)))
    print(method, {"paired_episode_count": paired_episode_count, "expected_pairs": N_MODEL_RUNS * N_EVAL_STARTS, **student_t_summary(run_differences)})

print("Initial-plan exploitability gap, summarized by independent model run")
for method in ("cem_mean", "cem_risk"):
    run_gaps = []
    missing_plans = 0
    for run_record in model_runs:
        rows = [row for row in plan_records if row["method"] == method and row["model_seed"] == run_record["model_seed"] and row["status"] == "complete"]
        missing_plans += N_EVAL_STARTS - len(rows)
        if rows:
            run_gaps.append(float(np.mean([float(row["exploitability_gap"]) for row in rows])))
    print(method, {"missing_plan_rows": missing_plans, **student_t_summary(run_gaps)})

RUN_FIELDS = ["model_seed", "training_episode_seeds", "bootstrap_seeds", "training_episode_count", "training_sample_count", "training_collection_seconds", "model_fit_seconds", "training_elapsed_seconds", "calibration_seconds", "final_evaluation_seconds", "elapsed_seconds", "model_config", "software_versions", "platform", "calibration_status", "status", "error_message"]
run_csv_rows = []
for row in model_runs:
    run_csv_rows.append({
        "model_seed": row["model_seed"],
        "training_episode_seeds": json.dumps(row["training_episode_seeds"]),
        "bootstrap_seeds": json.dumps(row["bootstrap_seeds"]),
        "training_episode_count": row["training_episode_count"],
        "training_sample_count": row["training_sample_count"],
        "training_collection_seconds": row["training_collection_seconds"],
        "model_fit_seconds": row["model_fit_seconds"],
        "training_elapsed_seconds": row["training_elapsed_seconds"],
        "calibration_seconds": row["calibration_seconds"],
        "final_evaluation_seconds": row["final_evaluation_seconds"],
        "elapsed_seconds": row["elapsed_seconds"],
        "model_config": json.dumps({"polynomial_degree": POLY_DEGREE, "ridge_alpha": RIDGE_ALPHA, "bootstrap_members": N_BOOTSTRAP_MEMBERS}),
        "software_versions": json.dumps(SOFTWARE_VERSIONS, sort_keys=True),
        "platform": platform.platform(),
        "calibration_status": row["calibration_status"],
        "status": row["status"],
        "error_message": row["error_message"],
    })

CALIBRATION_FIELDS = ["model_seed", "split", "episode_seed", "step", "component", "action", "previous_position", "previous_velocity", "observed_next_state", "ensemble_mean", "ensemble_std", "interval_scale", "interval_low", "interval_high", "covered", "interval_width"]
PLAN_FIELDS = ["model_seed", "evaluation_seed", "method", "start_position", "start_velocity", "action_sequence", "predicted_mean_return", "ensemble_return_std", "planner_score", "simulator_open_loop_return", "simulator_open_loop_steps", "exploitability_gap", "planner_model_calls", "planner_seconds", "status", "failure_detail"]
EPISODE_FIELDS = ["method", "model_seed", "evaluation_seed", "status", "success", "episode_return", "steps", "model_calls", "planner_seconds", "elapsed_seconds", "failure_detail"]
write_csv(OUTPUT_DIR / "model-uncertainty-runs.csv", run_csv_rows, RUN_FIELDS)
write_csv(OUTPUT_DIR / "model-uncertainty-calibration.csv", calibration_records, CALIBRATION_FIELDS)
write_csv(OUTPUT_DIR / "model-uncertainty-plans.csv", plan_records, PLAN_FIELDS)
write_csv(OUTPUT_DIR / "model-uncertainty-episodes.csv", episode_records, EPISODE_FIELDS)

print("Evidence rows written:", {
    "runs": len(run_csv_rows),
    "calibration": len(calibration_records),
    "plans": len(plan_records),
    "episodes": len(episode_records),
})
print("Complete-comparison gate:", all(row["status"] == "complete" for row in model_runs) and all(row["status"] == "complete" for row in plan_records) and all(row["status"] == "complete" for row in episode_records))
print("Notebook experiment-cell elapsed seconds:", round(perf_counter() - NOTEBOOK_STARTED, 3))

Closed-loop return: per-run means, then summary across model runs
cem_mean {'observed_runs': 5, 'missing_episode_rows': 0, 'n': 5, 'mean': -1.597982344725763, 'median': -1.547647368871501, 'lower_95': -1.8106169745868859, 'upper_95': -1.3853477148646403}
cem_risk {'observed_runs': 5, 'missing_episode_rows': 0, 'n': 5, 'mean': -1.59901218192716, 'median': -1.5303915169307547, 'lower_95': -1.8088025888464785, 'upper_95': -1.3892217750078417}
direct_momentum {'observed_episodes': 20, 'expected_episodes': 20, 'mean_return': 91.975, 'median_return': 92.00000000000001}
Observed success rate
cem_mean {'missing_episode_rows': 0, 'n': 5, 'mean': 0.0, 'median': 0.0, 'lower_95': 0.0, 'upper_95': 0.0}
cem_risk {'missing_episode_rows': 0, 'n': 5, 'mean': 0.0, 'median': 0.0, 'lower_95': 0.0, 'upper_95': 0.0}
direct_momentum {'observed_episodes': 20, 'expected_episodes': 20, 'success_rate': 1.0}
Paired return differences (learned method minus direct momentum, same evaluation seed)
cem_mean {'paired_e

Evidence rows written: {'runs': 5, 'calibration': 60000, 'plans': 200, 'episodes': 220}
Complete-comparison gate: True
Notebook experiment-cell elapsed seconds: 102.963


## What this result can show

This experiment can show how one small state-based model behaves on one controlled task. It can show whether calibration changes under planner actions and whether the model overpredicts its own selected plans. It cannot establish that the same result holds for image-based JEPA models, other tasks, physical robots, or the cited paper's experiments. The 90% target is a nominal calibration level, not a guaranteed outcome.

## Research context

- Yu et al., [Decision-Centric Evaluation of World Models](https://arxiv.org/html/2606.15032), discuss policy-induced shift, uncertainty calibration, and model-exploitability measures. This notebook borrows evaluation questions, not their experimental result.
- The environment follows the [Gymnasium 0.29.1 MountainCarContinuous implementation](https://github.com/Farama-Foundation/Gymnasium/blob/v0.29.1/gymnasium/envs/classic_control/continuous_mountain_car.py). This notebook uses a 200-step cap rather than the environment's 999-step default.
- The separate [EnvProbe preprint](https://arxiv.org/html/2606.31422) studies budgeted environment probing for language-model agents. It is adjacent work, not evidence for learned embodied dynamics.

## Check your understanding

1. Why does calibration on random-action episodes not guarantee calibration for a planner that searches for high predicted return?
2. What does a positive exploitability gap mean in this notebook?
3. Which changes would be needed before calling this a JEPA paper reproduction?